# 1. Torchrun & Basics

`torchrun` is PyTorch's launcher utility for distributed training. It runs the **same script N times**, spawning one process per replica. Each process gets its own set of environment variables (`LOCAL_RANK`, `RANK`, `WORLD_SIZE`) that let it know its identity within the job — without you having to pass any of this manually.

## How it works

```
                        torchrun
                            │
              ┌─────────────┼─────────────┬─────────────┐
              │             │             │             │
          Process 0     Process 1     Process 2     Process 3
              │             │             │             │
          train.py      train.py      train.py      train.py
              │             │             │             │
            GPU 0         GPU 1         GPU 2         GPU 3
```

Each process is a fully independent Python interpreter running the exact same script — `torchrun` doesn't fork or share memory between them, it just launches N separate instances and injects the right environment variables into each one before execution.

## Environment variables per process

| Variable      | Process 0 | Process 1 | Process 2 | Process 3 |
|---------------|-----------|-----------|-----------|-----------|
| `LOCAL_RANK`  | 0         | 1         | 2         | 3         |
| `RANK`        | 0         | 1         | 2         | 3         |
| `WORLD_SIZE`  | 4         | 4         | 4         | 4         |

- **`LOCAL_RANK`** — the process's index *within the current machine*. Used to pick which GPU that process owns (`torch.cuda.set_device(local_rank)`).
- **`RANK`** — the process's *global* index across the entire job (all machines combined). Used by `init_process_group` to identify the process within the distributed group.
- **`WORLD_SIZE`** — total number of processes across *all* machines in the job.

> On a single-node job, `LOCAL_RANK` and `RANK` are identical, since there's only one machine. They diverge once you scale to multiple nodes (see below).

## Command line syntax

```bash
torchrun [options] script.py [script args]
```

## Key options

| Flag | Meaning |
|---|---|
| `--nproc-per-node` | Number of processes to spawn **on this machine** (typically = number of GPUs on that machine). This is a per-node count, not a global one. |
| `--nnodes` | Total number of machines participating in the job. |
| `--node_rank` | Identifies **which machine** this is among the `nnodes` (0-indexed). Must be set manually and differently on each machine — it is not auto-detected. |
| `--master_addr` | IPv4 address (4 octets, e.g. `192.168.1.10`) of the machine hosting rank 0. This is where all processes connect to perform the initial rendezvous. |
| `--master_port` | TCP port used for that rendezvous connection. Any free port above 1024 works; must be open in the firewall between nodes. |

## How global rank is computed

```
global_rank = node_rank * nproc_per_node + local_rank
```

Example: 2 nodes, 4 GPUs each (`nnodes=2`, `nproc_per_node=4`):

```
Node 0 (node_rank=0)
  local_rank 0 → global rank 0
  local_rank 1 → global rank 1
  local_rank 2 → global rank 2
  local_rank 3 → global rank 3

Node 1 (node_rank=1)
  local_rank 0 → global rank 4
  local_rank 1 → global rank 5
  local_rank 2 → global rank 6
  local_rank 3 → global rank 7
```

## Single-node example

```bash
torchrun --nproc-per-node=4 train.py
```

No `master_addr`/`master_port`/`node_rank` needed — with a single node, `torchrun` resolves rendezvous locally by default. For extra reliability, `--standalone` can be used to force a local-only rendezvous on an automatically chosen free port:

```bash
torchrun --standalone --nproc-per-node=4 train.py
```

## Multi-node example

Run on **each** machine, changing only `--node_rank`:

```bash
# Node 0 (master, IP 192.168.1.10)
torchrun --nproc-per-node=4 --nnodes=2 --node_rank=0 \
  --master_addr=192.168.1.10 --master_port=29500 train.py

# Node 1
torchrun --nproc-per-node=4 --nnodes=2 --node_rank=1 \
  --master_addr=192.168.1.10 --master_port=29500 train.py
```

`master_addr` is the **same** on both commands — it always points to the machine hosting rank 0, regardless of which node is executing the command.

In [ ]:
torchrun --nproc-per-node=4 --nnodes=2 --node_rank=0 --master_addr=120.08.07 --master_port=29007 train.py

to get acess of the actual `local_rank`, `rank`, `world_size` in the process you can get using the `os lib`

In [ ]:
import os

rank = int(os.environ["RANK"])
local_rank = int(os.environ["LOCAL_RANK"])
world_size = int(os.environ["WORLD_SIZE"])


# 2. GPU

After you create all the process, you need to pass each process to each GPU, cuz if you don't do that, all the process will run in the same GPU and get the `OOF PROBLEM`

To do that, it's simple. Let's see how PyTorch and Cuda organize the GPUs

**The Cuda enumarate each GPU from 0 to N**
```
GPU 0
GPU 1
GPU 2
GPU 3
```

So to put something in that GPU it's like we know `Model().to(device)` wheres the device it's `device = torch.device('cuda:NUMBER')`. Knowing that we can thing that the number of GPUs can be the same of the ranks, so we just pass the `local_rank` as the number of the GPU, so each GPU will have a different process

```python

local_rank = int(os.environ["LOCAL_RANK"])
device = torch.device(f'cuda:{local_rank}')
# And now pass
model = Model().to(device)
```

or you can do with the cuda method

```python

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
# And now pass
model = Model().to("cuda")
```

Now each process are in the different GPU, and solving the OOM problem

# 2. Process Group Inicialization

When we initialize processes, they are all independent. In order for them to work in a distributed manner, it is necessary for them to discover each other's existence and organize themselves into a process group, allowing communication and the exchange of information between them. So we use the `init_process_group()` to do that

The `backend` argument is the implementation responsible for communication between processes. He is the one who sends and receives the data when you do distributed operations.

`init_process_group(backend='')` Normally we uses the NCCL

> NCCL is the library responsible for fast communication between NVIDIA GPUs during distributed training.

And after doing the link of all the GPUs we can do some operations like:

```python
dist.all_reduce(tensor)
dist.broadcast(tensor, src=0)
dist.all_gather(output, input)
dist.barrier()
```

So basically, we need to connect all the processes through a single backend so they can communicate with each other. And the function do that


In [ ]:
import torch.distributed as dist

dist.init_process_group(backend='nccl')

We can also destroy the process group when we're done by calling `dist.destroy_process_group()`. This releases the distributed communication resources and helps prevent issues when the program exits or when creating a new process group later.

In [ ]:
dist.destroy_process_group()

# 3. DPP Function

Now that we've connected all the processes and GPUs, we can perform the distributed operations we discussed earlier. However, implementing these operations manually would take a lot of work. Instead of calling `broadcast()` ourselves to ensure that every process starts with the same model weights, and `all_reduce()` to average the gradients across all processes, we can simply use `DistributedDataParallel()`, which handles these operations automatically.

```python 
from torch.nn.parallel import DistributedDataParallel as DDP

model = Model()
model = DDP(model, device_ids=[local_rank])
```

Doing that we need to pass some arguments
- `model` = It's the model that we gonna administrated, syncronize the gradients and the hooks in the `backward()`
- `device_ids`= Each process have their own model, so we need to pass the local_rank to all the processes have the same model


> Buffers are not updated as we go, so we should use SyncBatch in some cases

In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 10),
    nn.Relu()
)

model = DDP(model, device_ids=[local_rank])

# 4. Data Function

Now that each process have their GPU and the same model, we need to remember the main objective of the Data Parallel, that's train the same model in differents GPUs and get the medium gradient to update, but the data in each process need to be different cuz if it's the same will be useless, so we need to use an function to do each process have a different data. So we use the `DistributedSampler()`

And the arguments
- `dataset` = The dataset, needed to get acess of the indexs
- `num_replicas` = It's the world_size, will split in groups of indexs
- `rank` = It's the actual local_rank, need to get the group of the index

And we pass this sampler for the loader

```python

from torch.utils.data.distributed import DistributedSampler  # Import the sampler

sampler  = DistributedSampler(
    dataset = dataset,
    rank = local_rank,
    num_replicas = world_size,
    shuffle = True
)

loader = DataLoader(
    dataset = dataset,
    sampler = sampler,
    batch_size = 32 # THIS BATCH_SIZE WILL BE THE NUMBER FOR EACH GPU (IF 4 GPUS = 4 * 32)
    shuffle = False # If you use shuffle in the sampler you can't use in the loader
)
```

In [ ]:
rank = torch.distributed.get_rank() 
world_size = torch.distributed.get_world_size()
from torch.utils.data.distributed import DistributedSampler

sampler = DistributedSampler(
    dataset, 
    num_replicas=world_size,
    rank=rank,
    shuffle=True
    )


dataloader = DataLoader(
    dataset,
    batch_size=32,     
    sampler=sampler,    
    shuffle=False,      
    pin_memory=True
    )

# 5. Checkpoint

Like ever train, if something went wrong and lose everthing, we save in checkpoints. In this case we only need to save in 1 process, cuz if it wouldn't save in each process and that would be a problem


In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP
import torch

model = nn.Sequential(
    nn.Linear(10, 10),
    nn.Relu()
)

model = DDP(model, device_ids=[local_rank])

if dist.get_rank() == 0:
    torch.save(model.state_dict(), "checkpoint.pth")